In [116]:
import csv
EXPECTED_COLS = ["trip_id", "delay", "timestamp", "temperature", "precipitation", "rain", "showers", "snowfall"]

def load_data(path):
    with open(path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        assert header[: len(EXPECTED_COLS)] == EXPECTED_COLS, f"unexpected header: {header}"
        rows = [row[: len(EXPECTED_COLS)] for row in reader]
    df = pd.DataFrame(rows, columns=EXPECTED_COLS)
    df["delay"] = df["delay"].astype(int)
    for col in ["temperature", "precipitation", "rain", "showers", "snowfall"]:
        df[col] = df[col].astype(float)
    return df

In [117]:
import pandas as pd
import numpy as np


df = load_data("/Users/jettmu/Documents/VSCode/GTFS Parser/prediction-model/combined.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["date"] = df["timestamp"].dt.date


df["hour"] = df["timestamp"].dt.hour

df



,trip_id,delay,timestamp,temperature,precipitation,rain,showers,snowfall,date,hour
0,1984724,74,2026-08-13 02:34:41+00:00,18.6,0.0,0.0,0.0,0.0,2026-08-13,2
1,1984302,0,2026-08-13 02:34:41+00:00,18.6,0.0,0.0,0.0,0.0,2026-08-13,2
2,1985170,56,2026-08-13 02:34:41+00:00,18.6,0.0,0.0,0.0,0.0,2026-08-13,2
3,1995594,0,2026-08-13 02:34:41+00:00,18.6,0.0,0.0,0.0,0.0,2026-08-13,2
4,1985173,0,2026-08-13 02:34:41+00:00,18.6,0.0,0.0,0.0,0.0,2026-08-13,2
...,...,...,...,...,...,...,...,...,...,...
986919,1999359,-127,2026-08-03 00:19:51+00:00,18.6,0.1,0.1,0.0,0.0,2026-08-03,0
986920,1986622,62,2026-08-03 00:19:51+00:00,18.6,0.1,0.1,0.0,0.0,2026-08-03,0
986921,1994006,-205,2026-08-03 00:19:51+00:00,18.6,0.1,0.1,0.0,0.0,2026-08-03,0
986922,1999556,-422,2026-08-03 00:19:51+00:00,18.6,0.1,0.1,0.0,0.0,2026-08-03,0


In [118]:
df_trip = pd.read_csv("/Users/jettmu/Documents/VSCode/GTFS Parser/static-gtfs/data/yrt_archive/trips.txt")


In [119]:
df_trip["trip_id"] = df_trip["trip_id"].astype(str)



In [120]:
df = df.merge(df_trip[["trip_id", "route_id"]], on = "trip_id", how="left")

In [121]:
df_601 = df[df["route_id"] == 601.0]
df_601

,trip_id,delay,timestamp,temperature,precipitation,rain,showers,snowfall,date,hour,route_id
10,1988815,0,2026-08-13 02:34:41+00:00,18.6,0.0,0.0,0.0,0.0,2026-08-13,2,601.0
25,1988895,250,2026-08-13 02:34:41+00:00,18.6,0.0,0.0,0.0,0.0,2026-08-13,2,601.0
70,1988906,0,2026-08-13 02:34:41+00:00,18.6,0.0,0.0,0.0,0.0,2026-08-13,2,601.0
86,1988894,580,2026-08-13 02:34:41+00:00,18.6,0.0,0.0,0.0,0.0,2026-08-13,2,601.0
94,1988812,174,2026-08-13 02:34:41+00:00,18.6,0.0,0.0,0.0,0.0,2026-08-13,2,601.0
...,...,...,...,...,...,...,...,...,...,...,...
986858,1990590,-3,2026-08-03 00:19:51+00:00,18.6,0.1,0.1,0.0,0.0,2026-08-03,0,601.0
986882,1990634,-3,2026-08-03 00:19:51+00:00,18.6,0.1,0.1,0.0,0.0,2026-08-03,0,601.0
986884,1990635,-20,2026-08-03 00:19:51+00:00,18.6,0.1,0.1,0.0,0.0,2026-08-03,0,601.0
986889,1990639,-136,2026-08-03 00:19:51+00:00,18.6,0.1,0.1,0.0,0.0,2026-08-03,0,601.0


In [122]:
rain = df.groupby(["date", "hour"])[["rain", "delay"]].mean()

In [123]:
rain = rain.sort_values("rain", ascending=False)
rain

rain      delay
date       hour                     
2026-08-02 17    2.389018 -26.183492
           12    0.546421 -16.987730
2026-07-27 20    0.525353  -4.916705
2026-08-02 16    0.284855   1.844098
           13    0.249845 -15.091963
...                   ...        ...
2026-07-30 12    0.000000 -10.610014
           11    0.000000 -13.904621
           10    0.000000 -12.363952
           9     0.000000 -17.886513
2026-08-20 14    0.000000 -14.042021

[399 rows x 2 columns]

In [124]:
rain_win = 1

df_sample = df_601[     (df["timestamp"].dt.day_of_week == pd.Timestamp("2026-08-02").day_of_week)    &   (df["hour"] >= 17 - rain_win)    &   (df["hour"] <= 17 + rain_win)]

/var/folders/85/_bl8r5fd2tbgc6v1kzl1xszr0000gn/T/ipykernel_60947/2358413984.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_sample = df_601[     (df["timestamp"].dt.day_of_week == pd.Timestamp("2026-08-02").day_of_week)    &   (df["hour"] >= 17 - rain_win)    &   (df["hour"] <= 17 + rain_win)]


In [125]:
df_sample

,trip_id,delay,timestamp,temperature,precipitation,rain,showers,snowfall,date,hour,route_id
260435,1990581,102,2026-08-16 16:04:57+00:00,23.9,0.0,0.0,0.0,0.0,2026-08-16,16,601.0
260436,1990559,0,2026-08-16 16:04:57+00:00,23.9,0.0,0.0,0.0,0.0,2026-08-16,16,601.0
260441,1990614,-1156,2026-08-16 16:04:57+00:00,23.9,0.0,0.0,0.0,0.0,2026-08-16,16,601.0
260477,1990613,92,2026-08-16 16:04:57+00:00,23.9,0.0,0.0,0.0,0.0,2026-08-16,16,601.0
260481,1990558,-1113,2026-08-16 16:04:57+00:00,23.9,0.0,0.0,0.0,0.0,2026-08-16,16,601.0
...,...,...,...,...,...,...,...,...,...,...,...
979257,1990621,22,2026-08-02 18:59:51+00:00,20.1,0.0,0.0,0.0,0.0,2026-08-02,18,601.0
979259,1990563,125,2026-08-02 18:59:51+00:00,20.1,0.0,0.0,0.0,0.0,2026-08-02,18,601.0
979260,1990624,-33,2026-08-02 18:59:51+00:00,20.1,0.0,0.0,0.0,0.0,2026-08-02,18,601.0
979268,1990565,-64,2026-08-02 18:59:51+00:00,20.1,0.0,0.0,0.0,0.0,2026-08-02,18,601.0


In [126]:
df_sample[["delay", "rain"]].corr()

,delay,rain
delay,1.00000,-0.00824
rain,-0.00824,1.00000


In [131]:
df_601['rain_1h']=df['rain'].rolling(24).sum()/ 3



In [128]:
df_sample = df_601[     (df_601["timestamp"].dt.day_of_week == pd.Timestamp("2026-08-02").day_of_week)    &   (df_601["hour"] >= 17 - rain_win)    &   (df_601["hour"] <= 17 + rain_win)]
df_sample[["delay", "rain_1h"]].corr()

,delay,rain_1h
delay,1.000000,-0.006429
rain_1h,-0.006429,1.000000


In [134]:
df_sample[df_sample['rain_1h'] > 0 & (df_sample['timestamp'].dt.date == pd.Timestamp("2026-08-02"))]

,trip_id,delay,timestamp,temperature,precipitation,rain,showers,snowfall,date,hour,route_id,rain_1h
975698,1990558,-147,2026-08-02 16:34:51+00:00,20.9,0.2,0.2,0.0,0.0,2026-08-02,16,601.0,0.533333
975727,1990559,-55,2026-08-02 16:34:51+00:00,20.9,0.2,0.2,0.0,0.0,2026-08-02,16,601.0,1.600000
975730,1990612,-481,2026-08-02 16:34:51+00:00,20.9,0.2,0.2,0.0,0.0,2026-08-02,16,601.0,1.600000
975732,1990561,-202,2026-08-02 16:34:51+00:00,20.9,0.2,0.2,0.0,0.0,2026-08-02,16,601.0,1.600000
975737,1990617,-70,2026-08-02 16:34:51+00:00,20.9,0.2,0.2,0.0,0.0,2026-08-02,16,601.0,1.600000
...,...,...,...,...,...,...,...,...,...,...,...,...
978774,1990563,58,2026-08-02 18:39:51+00:00,20.1,0.1,0.1,0.0,0.0,2026-08-02,18,601.0,0.800000
978780,1990565,-58,2026-08-02 18:39:51+00:00,20.1,0.1,0.1,0.0,0.0,2026-08-02,18,601.0,0.800000
978814,1990618,-114,2026-08-02 18:39:51+00:00,20.1,0.1,0.1,0.0,0.0,2026-08-02,18,601.0,0.800000
978827,1990568,-179,2026-08-02 18:44:51+00:00,20.1,0.0,0.0,0.0,0.0,2026-08-02,18,601.0,0.533333
